In [40]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
import ipympl
import matplotlib.pyplot as plt
%matplotlib ipympl
import seaborn as sns
import json
from pathlib import Path
import ncaa_bbStats as nb
import ncaa_bbStats.player_utils as pu
import os

In [72]:
MY_DATA_PATH = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/player_stats_cache")
master_csv_path = MY_DATA_PATH / "batting" / "batting_noMin.csv"

# Define column names
required_stats = [
    'g', 'ab', 'pa', 'h', '1b', '2b', '3b', 'hr', 'r', 'rbi', 
    'bb', 'so', 'hbp', 'sf', 'sh', 'gdp', 'sb', 'cs', 'avg', 
    'bb%', 'k%', 'bb/k', 'obp', 'slg', 'ops', 'iso', 'spd', 
    'babip', 'wsb', 'wrc', 'wraa', 'woba', 'wrc+'
]

# Define id columns
id_columns = ['year', 'name', 'team', 'team name'] 

try:
    # 1. Load data
    full_stats_df = pd.read_csv(master_csv_path, usecols=id_columns + required_stats)
    
    # 2. Filter for the 2010-2025 timeframe
    master_roster_df = full_stats_df[
        (full_stats_df['year'] >= 2010) & (full_stats_df['year'] <= 2025)
    ].copy()
    
    # 3. Clean strings to prevent join errors later
    master_roster_df['team name'] = master_roster_df['team name'].astype(str).str.strip()
    master_roster_df['name'] = master_roster_df['name'].astype(str).str.strip()
    
    # 4. Sort by team, player, season
    master_roster_df = master_roster_df.sort_values(['team name', 'name', 'year'])
    
    print(f"Successfully imported {len(master_roster_df)} records for the 2010-2025 period.")
    print(master_roster_df.head())

# This is the part that was missing!
except FileNotFoundError:
    print(f"Error: Could not find the file at {master_csv_path}")
except ValueError as e:
    print(f"Error: Column mismatch or data type issue: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully imported 26826 records for the 2010-2025 period.
                 name team                     team name  year   g   ab   pa  \
10004  Aaron Staehely  ACU  Abilene Christian University  2022  33   75   83   
20637  Aaron Staehely  ACU  Abilene Christian University  2024  45  122  142   
6396        Adam Byrd  ACU  Abilene Christian University  2022  21    3    3   
1248   Alexei Cazarin  ACU  Abilene Christian University  2021  48  110  132   
6529   Alexei Cazarin  ACU  Abilene Christian University  2022  36   54   58   

        h  1b  2b  ...       slg       ops       iso       spd     babip  \
10004  20  17   2  ...  0.333333  0.662602  0.066667  6.012170  0.322034   
20637  33  24   5  ...  0.393443  0.734022  0.122951  8.020768  0.348315   
6396    1   1   0  ...  0.333333  0.666667  0.000000  2.642857  0.500000   
1248   26  18   6  ...  0.336364  0.659198  0.100000  6.899919  0.324675   
6529   12   7   3  ...  0.370370  0.633528  0.148148  6.509723  0.297297   



In [73]:
PITCH_PATH = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/player_stats_cache/pitching")
pitch_csv = PITCH_PATH / "pitching_noMin.csv"

# 2. Define ID columns (matching your batting script)
id_columns = ['year', 'name', 'team', 'team name'] 

# 3. Define the Pitching Stats columns
pitching_stats = [
    'age', 'playerid', 'w', 'l', 'era', 'g', 'gs', 'cg', 'sho', 'sv', 'ip', 'tbf',
    'h', 'r', 'er', 'hr', 'bb', 'hbp', 'wp', 'bk', 'so',
    'k/9', 'bb/9', 'k/bb', 'hr/9', 'k%', 'bb%', 'k-bb%', 
    'avg', 'whip', 'babip', 'lob%', 'fip', 'e-f'
]

try:
    # 4. Load the data using the combined columns list
    # usecols=id_columns + pitching_stats merges the two lists
    master_pitching_df = pd.read_csv(pitch_csv, usecols=id_columns + pitching_stats)
    
    # 5. Filter for the 2010-2025 timeframe
    master_pitching_df = master_pitching_df[
        (master_pitching_df['year'] >= 2010) & (master_pitching_df['year'] <= 2025)
    ].copy()
    
    # 6. Sort by team, name, and year for continuity logic
    # This matches the sorting logic used for your hitters
    master_pitching_df = master_pitching_df.sort_values(['team name', 'name', 'year'])
    
    # 7. Data Cleaning: Ensure numeric types for modeling
    master_pitching_df['ip'] = pd.to_numeric(master_pitching_df['ip'], errors='coerce')
    master_pitching_df['era'] = pd.to_numeric(master_pitching_df['era'], errors='coerce')
    
    print(f"Successfully imported {len(master_pitching_df)} pitching records.")
    display(master_pitching_df.head())

except FileNotFoundError:
    print(f"Error: File not found at {pitch_csv}")
except ValueError as e:
    print(f"Error: Column mismatch. Check if all pitching_stats headers exist. {e}")

Successfully imported 25964 pitching records.


,name,team,team name,age,playerid,year,w,l,era,g,...,hr/9,k%,bb%,k-bb%,avg,whip,babip,lob%,fip,e-f
5997,Adam Byrd,ACU,Abilene Christian University,NaN,sa3028197,2022,0,3,6.375000,18,...,0.375000,0.342105,0.157895,0.184211,0.204545,1.500000,0.354167,0.563380,5.151049,1.223951
10920,Adam Byrd,ACU,Abilene Christian University,NaN,sa3028197,2023,0,0,11.454545,16,...,0.818182,0.283582,0.253731,0.029851,0.276596,2.727273,0.444444,0.569620,7.725479,3.729066
4354,Adam Stephenson,ACU,Abilene Christian University,NaN,sa3043040,2021,1,4,3.715596,15,...,0.247706,0.206061,0.151515,0.054545,0.241007,1.610092,0.312500,0.697074,4.502456,-0.786860
9424,Adam Stephenson,ACU,Abilene Christian University,NaN,sa3043040,2022,3,3,6.243750,15,...,1.012500,0.198413,0.130952,0.067460,0.296296,1.818750,0.362500,0.676856,6.221882,0.021868
14483,Adam Stephenson,ACU,Abilene Christian University,NaN,sa3043040,2023,5,3,5.213793,19,...,1.675862,0.336585,0.087805,0.248780,0.240642,1.303448,0.330275,0.674603,5.226420,-0.012627


In [74]:
# 1. Standardize the Team Column in Master Standings
# This ensures it matches the 'team name' header in your Player CSVs
master_teams_df = master_teams_df.rename(columns={'team': 'team name'})

# Clean whitespace to ensure perfect matches
for df in [master_teams_df, master_roster_df, master_pitching_df]:
    if 'team name' in df.columns:
        df['team name'] = df['team name'].astype(str).str.strip()

# 2. Aggregate Batting Stats to Team Level using 'team name'
team_batting = master_roster_df.groupby(['team name', 'year']).agg({
    'pa': 'sum',
    'hr': 'sum',
    'r': 'sum',
    'ops': 'mean',
    'wrc+': 'mean'
}).reset_index()

# 3. Aggregate Pitching Stats to Team Level using 'team name'
team_pitching = master_pitching_df.groupby(['team name', 'year']).agg({
    'ip': 'sum',
    'so': 'sum',
    'era': 'mean',
    'fip': 'mean',
    'whip': 'mean'
}).reset_index()

# 4. Merge Standings with Batting
complete_table = pd.merge(
    master_teams_df, 
    team_batting, 
    on=['team name', 'year'], 
    how='left'
)

# 5. Merge the result with Pitching
complete_table = pd.merge(
    complete_table, 
    team_pitching, 
    on=['team name', 'year'], 
    how='left'
)

print(f"Final Table Dimensions: {complete_table.shape}")
display(complete_table.head())

Final Table Dimensions: (14726, 62)


,Team,team name,league,G,W,L,T,BB (Batting),AB,H,...,pa,hr,r,ops,wrc+,ip,so,era,fip,whip
0,Florida St. (ACC),Florida St.,ACC,68,48,20,0,402,2291,688,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,New Mexico St. (WAC),New Mexico St.,WAC,60,36,23,1,382,2224,776,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Clemson (ACC),Clemson,ACC,70,45,25,0,375,2496,764,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NC State (ACC),NC State,ACC,62,38,24,0,333,2289,747,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arizona St. (Pac-12),Arizona St.,Pac-12,62,52,10,0,320,2181,734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
